# **Lab 10: CAPE and CIN**

In this lab, we will practice calculating and interpreting CAPE and CIN on Skew-Ts. THis lab has 19 questions, but don't be alarmed! There is not really any coding on your part, and most of this lab involves interpreting the Skew-Ts.

**Learning Target 15**: I can identify key parameters like the LCL, LFC, CAPE, and CIN from a Skew-T diagram..

## How to submit this lab

1. Check your code runs cleanly. In the top menu, click **Kernel > Restart Kernel and Run All Cells**.
2. Save your notebook by pressing `Cmd + S` on Mac or `Ctrl + S` on Windows, or by going to **File > Save Notebook**.
3. Go to **File > Save and export notebook as > PDF**. This should start a download of your lab in PDF form.
4. Open your PDF file to make sure everything looks right, and upload it to the Canvas assignment for today.
5. Make sure to close your kernels on the JupyterHub. On the top bar, click **Kernel > Shut Down All Kernels...** and confirm.

# Setup

Check that you are running with the aos330 environment.

In [ ]:
# Import core packages using standard nicknames
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import os

os.environ["PROJ_LIB"] = "/opt/anaconda3/envs/aos330/share/proj"
os.environ["PROJ_DATA"] = "/opt/anaconda3/envs/aos330/share/proj"

from metpy.plots import SkewT
from metpy.units import units
import metpy.calc

# Part 1: CAPE, CIN, and buoyancy (6 pts)

Start by reading this 18Z sounding from North Platte, Nebraska on 2026-07-20. This is a mid-day sounding in local time.

I will convert them into a form `metpy` accepts for you (arrays with the units attached). Following this, you can use these objects:
- `pres`: pressure in hectopascals
- `temp`: temperature in degrees Celsius
- `dewpt`: dewpoint in degrees Celsius

In [ ]:
# read in the radiosonde data
df = pd.read_csv("data/20260720_northplatte.csv")

# convert these to a format that metpy can read, including the units
pres = df.pressure.values * units("hPa")
temp = df.temperature.values * units("degC")
dewpt = df.dewpoint.values * units("degC")

# print output to see what it looks like
print(pres)
print(temp)
print(dewpt)

As usual, let's plot the Skew-T using `metpy`. Below, the isotherms and isobars are gray solid lines, the dry adiabats are red dashed lines, the moist adiabats are blue dashed lines, and the mixing lines are green dashed lines. The solid black lines are the plotted sounding's dewpoint (left) and temperature (right).

In [ ]:
fig = plt.figure(figsize=(9, 9))

# initialize the Skew-T axis
skew = SkewT(fig, rotation=45)

# axis labels
skew.ax.set_xlabel("Temperature (degC)")
skew.ax.set_ylabel("Pressure (hPa)")

skew.ax.plot(temp, pres, color="black", lw=2)
skew.ax.plot(dewpt, pres, color="black", lw=2)

# add dry adiabat in the red dashed lines
skew.plot_dry_adiabats(colors="red")

# add moist adiabats in the blue dashed lines
skew.plot_moist_adiabats(colors="blue")

# add mixing lines in the green dashed lines
skew.plot_mixing_lines(colors="green")

plt.show()

### Question 1 (1 pt)

Use the Skew-T to estimate the surface pressure, temperature, and dewpoint. 

Estimate to within 2degC for the temperature/dewpoint and within 20 hPa for the pressure.

[1/3 point each value]

#### Answer 1

#TODO: *(Type your answer here)*

### Question 2 (1 pt)

Use the graphical method to find the LCL, and answer the following questions:

1. At what pressure would the parcel become saturated? 
2. Does the LCL depend on the static stability of the profile above the surface? Why or why not?

Reminder on steps for finding the LCL:
1. Find the parcel's surface temperature and dewpoint.
2. Follow the dry adiabat up from the surface temperature.
3. Follow the mixing ratio line up from the surface dewpoint.
4. Find the intersection of the dry adiabat and mixing ratio line. 
5. Look along a horizontal line from that intersection to the y-axis labels to read the pressure value of the LCL. 

[0.5 pt for right pressure; 0.5 pt for explanation about LCL]

#### Answer 2

#TODO: *(Type your answer here)*

To make our lives easier, we can also plot the trajectory of a surface-based parcel using `metpy` (feel free to use this to check you answers in 1 and 2 if you need to).

Below, I've plotted the **parcel trajectory** in the thick purple line. Notice that below the LCL, the trajectory follows a dry adiabat, and above the LCL, it follows a moist adiabat.

In [ ]:
fig = plt.figure(figsize=(9, 9))

# initialize the Skew-T axis
skew = SkewT(fig, rotation=45)

# axis labels
skew.ax.set_xlabel("Temperature (degC)")
skew.ax.set_ylabel("Pressure (hPa)")

skew.ax.plot(temp, pres, color="black", lw=2)
skew.ax.plot(dewpt, pres, color="black", lw=2)

# add dry adiabat in the red dashed lines
skew.plot_dry_adiabats(colors="red")

# add moist adiabats in the blue dashed lines
skew.plot_moist_adiabats(colors="blue")

# add mixing lines in the green dashed lines
skew.plot_mixing_lines(colors="green")

# NEW: calculate and plot the surface-based parcel trajectory
surf_temp = temp[0]
surf_dewpoint = dewpt[0]
parcel_profile = metpy.calc.parcel_profile(
    pres,
    # notice that this function ONLY accepts as inputs the initial temperature and dewpoint, rather than the whole profile
    surf_temp,
    surf_dewpoint,
)
skew.ax.plot(
    parcel_profile,
    pres,
    color="purple",
    lw=4,
)

# shade the area corresponding to CAPE and CIN
skew.shade_cape(pres, temp, parcel_profile)
el_p, _ = metpy.calc.el(pres, temp, dewpt, parcel_profile)
troposphere = (
    pres >= el_p
)  # masking out negative buoyant areas above the equilibrium level
skew.shade_cin(pres[troposphere], temp[troposphere], parcel_profile[troposphere])

plt.show()
plt.close("all")

### Question 3 (1 pt)

1. How does the surface-based parcel trajectory (in purple) compare to the environmental sounding (right black line) from the surface to ~800 hPa? Describe whether the parcel is warmer, cooler, or the same temperature as the environment at these levels.
2. Based on your answer above, do you expect the surface-based parcel to be positively, negatively, or neutrally buoyant?

[0.5 pts each]

#### Answer 3

#TODO: *(Type your answer here)*

### Question 4 (1 pt)

1. Right above 800 hPa, would the parcel be positively, negatively, or neutrally buoyant?
2. Keep following the parcel trajectory upwards. At what pressure level does the parcel's buoyancy relative to the environment change **for the first time**? This is the **level of free convection** or **LFC**.

[0.5 pts each]

#### Answer 4

#TODO: *(Type your answer here)*

## Question 5 (1 pt)

Above the LFC, the parcel becomes warmer than the environment and thus positively buoyant again. Keep following the parcel trajectory upwards. 

1. How far does this positively buoyant layer extend? The top of this layer is called the **equilibrium level** or **EL**, sometimes also called the **level of neutral buoyancy** or **LNB**.

2. If the parcel could make it above the LFC, would it start accelerating upwards or downwards?

[0.5 pts each]

#### Answer 5

#TODO: *(Type your answer here)*

## Question 6 (1 pt)

How do the negatively and positively buoyant regions of the parcel relate to CAPE and CIN? (Which region corresponds to what quantity?)

[1 pt for matching both pairs correctly]

#### Answer 6

#TODO: *(Type your answer here)*

Remember that CAPE and CIN are both *specific energies*, and so have units of J/kg.

We can use functions in `metpy` to compute the CAPE and CIN in this sounding as well! Use the function: `metpy.calc.surface_based_cape_cin(pres, temp, dewpt)` to calculate CAPE and CIN. This function numerically integrates the buoyancy in the parcel profile.

Note that the `metpy` function uses virtual temperature rather than just temperature, so the values are close to but not exactly proportional to the area in the curve as described in the Petty textbook. Also, the `metpy` function omits the negative sign for CIN, so it reports CIN as a negative number. 

In [ ]:
cape, cin = metpy.calc.surface_based_cape_cin(pres, temp, dewpt)

print(f"The CAPE in this profile is: {cape}.")
print(f"The CIN in this profile is: {cin}.")

# Part 2: Comparing non-local instability visually (7 pts)

In this part, we will practice comparing values of CAPE and CIN visually. 

Below, I'll plot the original North Platte sounding on the left, and a new sounding from Murcia, Spain on the right. 

In [ ]:
fig = plt.figure(figsize=(18, 8))

# define two sites
sites = ["20260720_northplatte", "20260617_murcia"]

# loop over sites
for i, site in enumerate(sites):
    # read in sounding
    df = pd.read_csv(f"data/{site}.csv")

    # convert these to a format that metpy can read, including the units
    pres = df.pressure.values * units("hPa")
    temp = df.temperature.values * units("degC")
    dewpt = (df.dewpoint.values) * units("degC")

    # initialize the Skew-T axis
    skew = SkewT(fig, rotation=45, subplot=(1, len(sites), i + 1))

    # label with site name
    skew.ax.set_title(site)

    # axis labels
    skew.ax.set_xlabel("Temperature (degC)")
    skew.ax.set_ylabel("Pressure (hPa)")

    skew.ax.plot(temp, pres, color="black", lw=2)
    skew.ax.plot(dewpt, pres, color="black", lw=2)

    # add dry adiabat in the red dashed lines
    skew.plot_dry_adiabats(colors="red")

    # add moist adiabats in the blue dashed lines
    skew.plot_moist_adiabats(colors="blue")

    # add mixing lines in the green dashed lines
    skew.plot_mixing_lines(colors="green")

    # calculate and plot the surface-based parcel trajectory
    surf_temp = temp[0]
    surf_dewpoint = dewpt[0]
    parcel_profile = metpy.calc.parcel_profile(
        pres,
        # notice that this function ONLY accepts as inputs the initial temperature and dewpoint, rather than the whole profile
        surf_temp,
        surf_dewpoint,
    )
    skew.ax.plot(
        parcel_profile,
        pres,
        color="purple",
        lw=4,
    )

    # shade the area corresponding to CAPE and CIN
    skew.shade_cape(pres, temp, parcel_profile)
    el_p, _ = metpy.calc.el(pres, temp, dewpt, parcel_profile)
    troposphere = (
        pres >= el_p
    )  # masking out negative buoyant areas above the equilibrium level
    skew.shade_cin(pres[troposphere], temp[troposphere], parcel_profile[troposphere])


plt.show()
plt.close("all")

## Question 7 (1 pt)

Based on these Skew-Ts, estimate the level of free convection and the equilibrium level in the Murcia sounding.

[0.5 pts per value]

#### Answer 7

#TODO: *(Type your answer here)*

## Question 8 (1 pt)

Compare these two soundings in terms of the *magnitudes* of CAPE and CIN. 

[0.5 pts for CAPE comparison, 0.5 pts for CIN comparison]

#### Answer 8

#TODO: *(Type your answer here)*

## Question 9 (2 pts)

For each sounding, how does the amount of CAPE compare to the amount of CIN? Which sounding has greater non-local instability?

[0.5 pts per sounding description; 1 pt for assessing non-local instability]

#### Answer 9

#TODO: *(Type your answer here)*

## Question 10 (1 pt)

Based on your answer in Question 9, which sounding do you think would be more likely to support deep convection? Explain your reasoning briefly.

[1 pt for explanation]

#### Answer 10

#TODO: *(Type your answer here)*

## Question 11 (2 pts)

Look closely at the environmental temperature profile in the Murcia sounding, focusing on the region where the parcel trajectory is negatively buoyant. How would you describe the lower part of this layer in terms of its static stability? Describe how its temperature profile sets up the amount of CIN you see in that sounding.

You may find it helpful to compare this profile to the North Platte one at the same levels.

[1 pt for stability description, 1 pt for impact on CAPE]

#### Answer 11

#TODO: *(Type your answer here)*

# Part 3: Shapes of CAPE

Remember that although CAPE is very useful for determining the energy available for a lifted parcel, it is ultimately an *integrated* metric (i.e., added up over different heights). In other words, profiles of different shapes can still produce similar CAPE values.

In this part, we will compare three soundings:

1. Koror, Palau (island country in the Western Pacific) at 0Z 2026-05-12 (morning sounding)
2. Norman, Oklahoma at 12Z 2026-07-04 (early morning sounding on a summer day where thunderstorms later developed)
3. Fort Worth, Texas at 12Z 2026-03-07 (early morning sounding on spring day warned for enhanced risk of severe weather)

This time, I'll annotate the values of CAPE and CIN so you can read them directly.

In [ ]:
fig = plt.figure(figsize=(24, 8))

# loop over the three sites
sites = ["20260307_fortworth", "20260704_norman", "20260512_palau"]
for i, site in enumerate(sites):
    # read in sounding
    df = pd.read_csv(f"data/{site}.csv")

    # convert these to a format that metpy can read, including the units
    pres = df.pressure.values * units("hPa")
    temp = df.temperature.values * units("degC")
    dewpt = (df.dewpoint.values) * units("degC")

    # initialize the Skew-T axis
    skew = SkewT(fig, rotation=45, subplot=(1, len(sites), i + 1))

    # label with site name
    skew.ax.set_title(site)

    # axis labels
    skew.ax.set_xlabel("Temperature (degC)")
    skew.ax.set_ylabel("Pressure (hPa)")

    skew.ax.plot(temp, pres, color="black", lw=2)
    skew.ax.plot(dewpt, pres, color="black", lw=2)

    # add dry adiabat in the red dashed lines
    skew.plot_dry_adiabats(colors="red")

    # add moist adiabats in the blue dashed lines
    skew.plot_moist_adiabats(colors="blue")

    # add mixing lines in the green dashed lines
    skew.plot_mixing_lines(colors="green")

    # NEW: calculate and plot the surface-based parcel trajectory
    surf_temp = temp[0]
    surf_dewpoint = dewpt[0]
    parcel_profile = metpy.calc.parcel_profile(
        pres,
        # notice that this function ONLY accepts as inputs the initial temperature and dewpoint, rather than the whole profile
        surf_temp,
        surf_dewpoint,
    )
    skew.ax.plot(
        parcel_profile,
        pres,
        color="purple",
        lw=4,
    )

    # shade the area corresponding to CAPE and CIN
    skew.shade_cape(pres, temp, parcel_profile)
    el_p, _ = metpy.calc.el(pres, temp, dewpt, parcel_profile)
    troposphere = (
        pres >= el_p
    )  # masking out negative buoyant areas above the equilibrium level
    skew.shade_cin(pres[troposphere], temp[troposphere], parcel_profile[troposphere])

    # NEW: calculate and plot CAPE and CIN values
    cape, cin = metpy.calc.surface_based_cape_cin(pres, temp, dewpt)

    skew.ax.text(
        0.04,
        0.04,
        f"CAPE: {cape.to('J/kg').magnitude:.0f} J/kg\nCIN: {cin.to('J/kg').magnitude:.0f} J/kg",
        transform=skew.ax.transAxes,
        fontsize=11,
        zorder=10,
        bbox=dict(facecolor="white"),
    )

plt.show()

## Question 12 (1 pts)

How do the magnitudes of CAPE compare across these three soundings? Compare the spread in CAPE here to the first pair of soundings you analyzed in Part 2.

[0.5 pt for describing magnitudes, 0.5 pt for comparison to Pt2]

#### Answer 12

#TODO: *(Type your answer here)*

## Question 13 (1 pt)

In which of these soundings would a surface-based parcel have the most difficult time "unlocking" the potential energy stored in buoyancy? Describe your reasoning.

[0.5 pts for answer, 0.5 pts for reasoning]

#### Answer 13

#TODO: *(Type your answer here)*

## Question 14 (1 pt)

For the sounding you identified in Question 13, describe a change that could occur to the sounding (e.g., say whether you would warm/cool/moisten/dry a particular layer of the atmosphere) in order to reduce the amount of CIN.

[1 pt for a plausible and clear description]

#### Answer 14

#TODO: *(Type your answer here)*

## Question 15 (3 pt)

Notice that although all three soundings contain a similar magnitude of CAPE, it is distributed quite differently.

1. Which of the three soundings has the biggest vertical distance between the LFC and the EL? Estimate the depth of this layer in hPa
2. In which sounding is the parcel most buoyant on average across this layer? Describe how you know this based on the graph. 
3. Explain why these three soundings have similar total CAPE even though they look quite different. 

[1 pt per question]

#### Answer 15

#TODO: *(Type your answer here)*

# Part 4: Breaking the cap (6 pts)

Finally, we are going to explore how the "cap" of negatively buoyant air that makes up CIN can be eroded over the course of the day. 

These soundings were taken in Jackson, Mississippi over the course of 11 hours from early morning to evening on June 19, 2023 (local time). This day was part of a multi-day period of severe weather and tornado outbreaks in the Southern US. Here, the `Z` in the title of each plot means the time in "Zulu" or coordinated universal time.

Note: The following plot is just the environmental sounding without the shaded area for CAPE/CIN and without the parcel trajectory, so you can practice reading the Skew-Ts by yourself.

In [ ]:
fig = plt.figure(figsize=(24, 8))

# loop over the three sites
sites = [
    "20230618_13Z_jackson",
    "20230618_21Z_jackson",
    "20230619_0Z_jackson",
]

for i, site in enumerate(sites):
    # read in sounding
    df = pd.read_csv(f"data/{site}.csv")

    # convert these to a format that metpy can read, including the units
    pres = df.pressure.values * units("hPa")
    temp = df.temperature.values * units("degC")
    dewpt = (df.dewpoint.values) * units("degC")

    # initialize the Skew-T axis
    skew = SkewT(fig, rotation=45, subplot=(1, len(sites), i + 1))

    # label with site name
    skew.ax.set_title(site)

    # axis labels
    skew.ax.set_xlabel("Temperature (degC)")
    skew.ax.set_ylabel("Pressure (hPa)")

    skew.ax.plot(temp, pres, color="black", lw=2)
    skew.ax.plot(dewpt, pres, color="black", lw=2)

    # add dry adiabat in the red dashed lines
    skew.plot_dry_adiabats(colors="red")

    # add moist adiabats in the blue dashed lines
    skew.plot_moist_adiabats(colors="blue")

    # add mixing lines in the green dashed lines
    skew.plot_mixing_lines(colors="green")


plt.show()

## Question 16 (2 pts)

Use the 13Z sounding to estimate:
1. surface temperature
2. height of the LCL (in hPa) 
3. height of the LFC (in hPa) 
4. height of the EL (in hPa) 

[0.5 pts per question]

#### Answer 16

#TODO: *(Type your answer here)*

## Question 17 (1 pt)

Describe how the surface temperature and LCL height evolve as the surface warms up during the daytime.

No need to quantify this, just describe whether they increase/decrease or go up/down in altitude.

[0.5 pts each]

#### Answer 17

#TODO: *(Type your answer here)*

## Question 18 (1 pt)

Describe how the LFC and CIN change over the three soundings. 

Again, no need to quantify this, just describe whether they increase/decrease or go up/down in altitude.

[0.5 pts each for LFC and CIN]

#### Answer 18

#TODO: *(Type your answer here)*

## Question 19 (2 pts)

Explain why you see these changes in CIN by describing the change in the position of the surface-based parcel versus the environmental temperature profile and what this means for buoyancy. From a non-local stability perspective, is the atmosphere getting more or less unstable over the course of the day?

[1 pt for change in parcel's buoyancy; 1 pt for impact on stability] 

### Answer 19

#TODO: *(Type your answer here)*